# Multimodal PDF RAG: Text and Image Evidence

| Field | Value |
|---|---|
| Stage | Multimodal RAG |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A multimodal pipeline keeps text and image evidence distinct, then fuses conclusions with page-level provenance instead of pretending every fact came from OCR text.

## 30-Second Summary

This notebook parses a one-page revenue PDF offline. Text extraction states that Q3 grew most; image analysis detects three colored bars with increasing heights. The modalities independently support the trend but neither provides exact revenue values.

## Why This Matters

Text-only parsing misses chart geometry, while image-only interpretation can miss captions and qualifications. Reliable answers need modality-aware records, provenance, and explicit limits.

## Scope

| Covers | Does not cover |
|---|---|
| PDF text, embedded-image extraction, simple bar geometry, modality fusion, page citations | General vision model, OCR, arbitrary chart understanding, exact unlabeled values |


## Mental Model

```text
PDF page -> text blocks --------\
           image assets -> vision/geometry -> fused evidence -> answer + page citation
```


In [1]:
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import fitz
from PIL import Image
from rag_101 import find_repo_root

REPO_ROOT = find_repo_root()
PDF_PATH = REPO_ROOT / "11-multiModal-multi-modal-rag/multimodal_sample.pdf"
pdf = fitz.open(PDF_PATH)
page = pdf[0]
len(pdf), len(page.get_text()), len(page.get_images(full=True))


(1, 365, 1)

## How It Works

We emit one text record and one image record from the same page. The image record carries dimensions, checksum, and parent page. A deliberately narrow geometry function finds the three saturated colored bars and compares heights; it is not a universal chart parser.


## Baseline

The text-only baseline can answer which quarter grew most because the prose says Q3 had the highest growth, but it cannot independently verify the visual trend.


In [2]:
page_text = page.get_text().strip()
text_record = {
    "id": "revenue:p1:text", "source": PDF_PATH.relative_to(REPO_ROOT).as_posix(),
    "page": 1, "modality": "text", "content": page_text,
}
{"characters": len(page_text), "mentions_q3_highest": "highest growth recorded in Q3" in page_text}


{'characters': 364, 'mentions_q3_highest': True}

## Technique Implementation

The image is extracted without saving a machine-specific path. Pixels are classified by dominant red, green, or blue channel; each color's vertical extent gives a bar height. The method is valid only for this fixture's simple unlabeled chart.


In [3]:
xref = page.get_images(full=True)[0][0]
image_payload = pdf.extract_image(xref)
image_bytes = image_payload["image"]
image = Image.open(BytesIO(image_bytes)).convert("RGB")
pixels = image.load()

def bar_height(channel: int) -> int:
    points = [
        (x, y) for y in range(image.height) for x in range(image.width)
        if pixels[x, y][channel] > 100 and all(pixels[x, y][other] < 80 for other in range(3) if other != channel)
    ]
    return max(y for _, y in points) - min(y for _, y in points) + 1

bar_heights = {"Q1": bar_height(2), "Q2": bar_height(1), "Q3": bar_height(0)}
image_record = {
    "id": "revenue:p1:image1", "source": text_record["source"], "page": 1,
    "modality": "image", "sha256": sha256(image_bytes).hexdigest(),
    "width": image.width, "height": image.height, "observations": bar_heights,
}
image_record


{'id': 'revenue:p1:image1',
 'source': '11-multiModal-multi-modal-rag/multimodal_sample.pdf',
 'page': 1,
 'modality': 'image',
 'sha256': '774ab01a109434cd1442e14df0149655e0c834a0ad941e254eefd4ec25f7813f',
 'width': 400,
 'height': 300,
 'observations': {'Q1': 101, 'Q2': 151, 'Q3': 201}}

## Controlled Experiment

We compare the prose claim with the image-derived ordering. Agreement supports the qualitative trend; absence of labels prevents a defensible numeric answer.


In [4]:
image_order = sorted(bar_heights, key=bar_heights.get)
text_claim = "Q3"
image_claim = image_order[-1]
results = {
    "text_highest_quarter": text_claim,
    "image_highest_quarter": image_claim,
    "modalities_agree": text_claim == image_claim,
    "bar_heights_pixels": bar_heights,
    "exact_revenue_available": False,
}
results


{'text_highest_quarter': 'Q3',
 'image_highest_quarter': 'Q3',
 'modalities_agree': True,
 'bar_heights_pixels': {'Q1': 101, 'Q2': 151, 'Q3': 201},
 'exact_revenue_available': False}

## Evaluation

Text and image agree that **Q3 is highest**, and the detected bar heights rise from Q1 to Q3. Because the chart has no labels or axis values, the system must not invent exact revenue numbers. The page citation remains `multimodal_sample.pdf`, page 1.


In [5]:
assert results["modalities_agree"]
assert image_order == ["Q1", "Q2", "Q3"]
assert all(value > 0 for value in bar_heights.values())
assert not results["exact_revenue_available"]
assert text_record["page"] == image_record["page"] == 1
pdf.close()
print("Multimodal extraction and fusion checks passed.")


Multimodal extraction and fusion checks passed.


## Decision Guide

| Evidence | Path |
|---|---|
| Narrative prose | Text parser/retriever |
| Labeled chart/table | Layout or chart extraction |
| Unlabeled visual trend | Vision/geometry with qualitative caveat |
| Cross-modal claim | Retrieve both and reconcile provenance |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Chart missing | Text-only ingestion | Extract/page-link images |
| Exact values invented | Unlabeled pixels treated as data | Abstain on numbers |
| Text and image disagree | Version/layout/extraction issue | Surface conflict and inspect source |
| Citation points to wrong page | Parent link dropped | Inherit source/page on every asset |


## Production Notes

### Observability
Track pages, text blocks, images, extraction method, modality agreement, model/version, and unresolved conflicts.

### Safety and Guardrails
Images can contain sensitive data and prompt-like text; apply the same authorization and redaction policy as prose.

### Latency and Cost
Route text-only questions away from vision; cache image features by content hash.


## Practice

Add axis labels to the fixture and specify what extra evidence is required before reporting numeric values.

## Recall

Toggle - Recall: Why keep modalities separate?
Their extraction methods, confidence, and failure modes differ.

Toggle - Recall: What can this image prove?
Only the relative bar ordering for this simple fixture, not exact revenue.

## Sources

- [PyMuPDF image extraction](https://pymupdf.readthedocs.io/en/latest/recipes-images.html)
- Repository fixture: `multimodal_sample.pdf`

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the fixture-specific qualitative check | Add labels, OCR, and cross-modal conflict fixtures |
